In [16]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.vectorstores import  FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface import ChatHuggingFace
from langchain_huggingface import HuggingFaceEndpoint
from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings
import numpy as np
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

In [2]:
## Reading the pdfs from folder

loader = PyPDFDirectoryLoader("./tam")

documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(documents)
docs[0]

Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-08-31T15:05:34+03:30', 'author': '2593920785', 'moddate': '2026-08-31T15:05:34+03:30', 'title': 'Microsoft Word - RC_IITP_IS_V7_9 (41405-06-09).docx', 'source': 'tam\\RC_IITP_IS_V7_9_2.pdf', 'total_pages': 116, 'page': 0, 'page_label': '1'}, page_content='ودیانड़ ଡی و ساماਘ\u0babو\u0dbc່ ی\u0b53 ଡظام پایاো تارජࡺग़ م࣓\u0a7bࣚಧ ච໊ජ໑  \n \n   \n  \n  \n  \n \nﺷﻨﺎﺳﻪ ﺳﻨﺪ: RC_IITP.IS _V7.9 \n  \n  \n  \n  \n \n  \nتير مﺎه ﺳﺎل ٠٤١٥')

In [3]:
len(docs)

743

In [4]:
## Embedding using huggingface
huggingface_embeddings = HuggingFaceEmbeddings(
    model_name="heydariAI/persian-embeddings",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

d:\AI Engineering\Langchain chat\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:02<00:00, 161.62it/s]


In [4]:
ollama_embedding = OllamaEmbeddings(model="qwen3-embedding:8b")

In [6]:
np.array(huggingface_embeddings.embed_query(docs[0].page_content))

array([-0.03916524, -0.01244748,  0.01581858, ..., -0.00555179,
        0.01341024,  0.00031923], shape=(1024,))

In [5]:
np.array(ollama_embedding.embed_query(docs[0].page_content))

array([ 0.01498901,  0.01790757, -0.00201685, ...,  0.00306239,
        0.01631222, -0.01903141], shape=(4096,))

In [8]:
## VectorStore Creation

vector_store = FAISS.from_documents(docs, huggingface_embeddings)

In [6]:
## VectorStore Creation

vector_store = FAISS.from_documents(docs, ollama_embedding)

In [7]:
## Query using Similarity Search

query = "مهلت ارسال صورت حساب کی؟"
relevant_documents = vector_store.similarity_search(query)
relevant_documents

[Document(id='0bec3b0d-2d64-413a-abf9-cb0e61bb3e93', metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-08-31T15:05:34+03:30', 'author': '2593920785', 'moddate': '2026-08-31T15:05:34+03:30', 'title': 'Microsoft Word - RC_IITP_IS_V7_9 (41405-06-09).docx', 'source': 'tam\\RC_IITP_IS_V7_9_2.pdf', 'total_pages': 116, 'page': 24, 'page_label': '25'}, page_content='فيلدهاي قاعده ارسال با مقدار يك و تاريخ و زمان ثبت صورتحساب مطابق قواعد (، صورتحساب به عنوان \nصورتحساب موضوع ماده )٩( قانون پايانه هاي فروشگاهي و سامانه موديان در نظر گرفته خواهد شد.  \n٧  \nاگر فاصله زماني تاريخ و زمان صدور صورتحساب تا ارسال آن در سامانه موديان بيشتر از مهلت مجاز اعﻼم شده \nتوسط سازمان امور مالياتي كشور باشد، در صورت عدم رعايت قواعد ارسال صورتحساب موضوع ماده )٩( )پر \nكردن فيلدهاي قاعده ارسال با مقدار يك و تاريخ و زمان ثبت صورتحساب مطابق قواعد(، صورتحساب  نامعتبر \nميگردد.  \n٨  تاريخ و زمان صدور صورتحساب از تاريخ و زمان حال نبايد بزرگتر باشد. )اولويت بررسي تاريخ سپس زمان 

In [8]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={'k': 10})
print(retriever)

tags=['FAISS', 'OllamaEmbeddings'] vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000023738048EF0> search_kwargs={'k': 10}


In [9]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ['HUGGINGFACEHUB_API_TOKEN'] = os.getenv('HUGGINGFACEHUB_API_TOKEN')
## langsmith tracking
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")

In [35]:
llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-R1-0528",
    task="text-generation",
    max_new_tokens=4096,
    do_sample=False,
    repetition_penalty=1.03,
    provider="auto"
    )

chat_model = ChatHuggingFace(llm=llm)

chat_model.invoke(query)

HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6b03c584-5b9f98f26070865e41f93c27;432df7d6-8b4e-407e-bdc7-0746beb51328)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get monthly included credits.

In [11]:
llm = ChatOllama(model="qwen3.8:27b")

llm.invoke(query)

AIMessage(content='سلام!\n\nبرای پاسخ دقیق به این سؤال، نیاز به اطلاعات بیشتری دارم:\n\n- **چه سازمان/شرکتی؟** (مثلاً یک شرکت خاص، سازمان دولتی، بیمه، دانشگاه و...)\n- **نوع صورت\u200cحساب؟** (فاکتور رسمی مالیاتی، صورت\u200cحساب پروژه، صورتحساب بیمه\u200cای، فاکتور خرید و...)\n- **چه دوره\u200cای؟** (ماهانه، فصلی، سالانه)\n\n---\n\n**به\u200cطور کلی در ایران:**\n\n| موضوع | مهلت معمول |\n|---|---|\n| ارسال فاکتور رسمی مالیاتی (به دريافت\u200cکننده) | معمولاً در همان ماه یا تا پایان ماه بعد |\n| ارسال اظهارنامه مالیاتی | اسفندماه (برای حقوقی) / فروردین (برای مشاغل) |\n| صورتحساب قراردادهای دولتی | طبق شرط قرارداد (معمولاً ۷ تا ۳۰ روز پس از تحویل) |\n| صورتحساب بیمه\u200cای | معمولاً ۱۰ تا ۱۵ روز کاری پس از پایان دوره |\n\n---\n\nاگر بفرمایید دقیقاً مربوط به کدام سازمان یا قرارداد است، راهنمایی دقیق\u200cتری ارائه می\u200cدهم. 🙂', additional_kwargs={}, response_metadata={'model': 'qwen3.8:27b', 'created_at': '2026-11-23T08:11:10.40199152Z', 'done': True, 'done_reason': 'stop', 'total_dur

In [ ]:
#Hugging Face models can be run locally through the HuggingFacePipeline class.
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline

hf = HuggingFacePipeline.from_model_id(
    model_id="mistralai/Mistral-7B-v0.1",
    task="text-generation",
    pipeline_kwargs={"temperature": 0, "max_new_tokens": 300}
)

llm = hf 
llm.invoke(query)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]d:\AI Engineering\Langchain chat\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Farahani.Mohamadd\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [36]:
promt_template = """
Use the following piece of context to answer the question asked.
Please try to provide the answer only based on the context

{context}
Question:{input}

Helpful Answers:
 """

In [37]:
prompt = PromptTemplate(template=promt_template, input_variables=["context", "input"])

In [38]:
retrievalQA=RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt":prompt}
)

In [42]:
result = retrievalQA.invoke({'input': query, 'query': query, 'input': query})
result['result']

ValueError: Missing some input keys: {'input'}

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '\n<html><head>\n<meta http-equiv="content-type" content="text/html;charset=utf-8">\n<title>403 Forbidden</title>\n</head>\n<body text=#000000 bgcolor=#ffffff>\n<h1>Error: Forbidden</h1>\n<h2>Your client does not have permission to get URL <code>/runs/multipart</code> from this server.</h2>\n<h2></h2>\n</body></html>\n')
